In [ ]:
import os
import sys
import random
import numpy as np
import cv2
import torch
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
from PIL import Image
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

In [ ]:
def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

In [ ]:
@dataclass
class CFG: 
    batch_size : int = 32
    seed: int = 42
    early_stopping_steps :int = 7
    number_of_channels : int = 3
    steps_until_plot: int = 2
    criterion : nn.Module = nn.CrossEntropyLoss(label_smoothing=0.1)
    epochs : int = 50
    device : str = 'mps' if torch.backends.mps.is_available() else 'cpu'
    img_size : int = 96
    base_path : str = 'imagebits'
    number_of_pictures_example: int = 5
    train_path : str = os.path.join(base_path, 'train')
    test_path : str = os.path.join(base_path, 'test')
    image_extension : str = '.png'
    mapping_between_folders_and_classes = {
        'airplane': '1',
        'bird': '2',
        'car': '3',
        'cat': '4',
        'deer': '5',
        'dog': '6',
        'horse': '7',
        'monkey': '8',
        'ship': '9',
        'truck': '10'
    }
    num_classes : int = len(mapping_between_folders_and_classes)
    imagenet_mean  = [0.485, 0.456, 0.406]
    imagenet_std  = [0.229, 0.224, 0.225]

In [ ]:
class_distribution = {}

for class_folder in sorted(os.listdir(CFG.train_path)):
    class_path = os.path.join(CFG.train_path, class_folder)
    if os.path.isdir(class_path):
        num_images = len([f for f in os.listdir(class_path) if f.endswith(CFG.image_extension)])
        class_distribution[class_folder] = num_images

In [ ]:
def remap_dictionary_keys_to_names_of_objects(original_dictionary):
    reversed_mapping = {v: k for k, v in CFG.mapping_between_folders_and_classes.items()}
    return {reversed_mapping.get(k, k): v for k, v in original_dictionary.items()}

new_dictionary = remap_dictionary_keys_to_names_of_objects(class_distribution)


In [ ]:
plt.figure(figsize=(10, 5))
classes = sorted(new_dictionary.keys(), key=lambda x: int(CFG.mapping_between_folders_and_classes[x]))
counts = [new_dictionary[c] for c in classes]
plt.bar(classes, counts, color='steelblue', edgecolor='black')
plt.xlabel('Clasa', fontsize=12)
plt.ylabel('Număr de imagini', fontsize=12)
plt.title('Distribuția Claselor în Setul de Antrenare', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
is_balanced = len(set(class_distribution.values())) == 1
print(f"Dataset-ul este {'Echilibrat' if is_balanced else 'Dezechilibrat'}")

In [ ]:
def get_parameters_for_type_of_operation(type = 'picture_stats'):
    if type == 'picture_stats':
        return [],{}
    elif type == 'correlation_purpose':
        return [],[]
    elif type == 'plot_images':
        return [],[]
    elif type == 'make_csv_files':
        return [],[]
    
    return None

In [ ]:
def get_random_images_to_plot(all_images):
    random.seed(CFG.seed)
    return random.sample(all_images, min(CFG.number_of_pictures_example, len(all_images)))

In [ ]:
def set_path(train=True):
    return CFG.train_path if train else CFG.test_path

In [ ]:
def get_path(class_idx,train=True):
    return os.path.join(set_path(train), str(class_idx))

In [ ]:
def get_all_images(class_idx,train=True):
    return [f for f in os.listdir(get_path(class_idx,train)) if f.endswith(CFG.image_extension)]

In [ ]:
def read_images(image_name,class_idx,train=True):
    img_path = os.path.join(get_path(class_idx,train),image_name)
    return img_path, cv2.imread(img_path)

In [ ]:
def prepare_plot():
    fig, axes = plt.subplots(CFG.num_classes, CFG.number_of_pictures_example, figsize=(16, 20))
    fig.suptitle('Exemple din Fiecare Clasă - Variabilitate Intra-Clasă', fontsize=16, fontweight='bold')
    return fig, axes

In [ ]:
def plot_examples_from_dataset(image,axes,class_idx,img_idx):
    ax = axes[class_idx - 1, img_idx]
    ax.imshow(image)
    ax.axis('off')
        
    if img_idx == 0:
        ax.set_title(f'Clasa {class_idx}', fontsize=12, fontweight='bold', loc='left')

In [ ]:
def process_all_images(number_of_classes=CFG.num_classes, operation_type='picture_stats', train=True):
    container1, container2 = get_parameters_for_type_of_operation(operation_type)

    if operation_type == 'plot_images':
        _, axes = prepare_plot()
    
    for class_idx in range(1, number_of_classes + 1):
        all_images = get_all_images(class_idx, train=train)

        if operation_type == 'plot_images':
            all_images = get_random_images_to_plot(all_images)
        
        class_pixels = [] 
        for img_name in all_images:
            img_path, image = read_images(img_name, class_idx, train=train)
            
            if image is None:
                continue
            
            if operation_type == 'picture_stats':
                container1.append(image.shape[:2]) 
                class_pixels.append(np.array(image))
                
            elif operation_type == 'correlation_purpose':
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                vector = image_rgb.flatten()
                container1.append(vector)
                container2.append(class_idx)
                
            elif operation_type == 'plot_images':
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                plot_examples_from_dataset(image_rgb, axes, class_idx, img_idx=all_images.index(img_name))

            elif operation_type == 'make_csv_files':
                container1.append({'ID': img_path, 'label': class_idx})
                
        if operation_type == 'picture_stats' and class_pixels:
            class_pixels_np = np.array(class_pixels)
            container2[class_idx] = {
                'mean': np.mean(class_pixels_np),
                'std': np.std(class_pixels_np),
                'min': np.min(class_pixels_np),
                'max': np.max(class_pixels_np)
            }

    if operation_type == 'correlation_purpose':
        return np.array(container1), np.array(container2)
    
    if operation_type == 'plot_images':
        plt.tight_layout()
        plt.show()
    
    return container1, container2

In [ ]:
image_sizes, pixel_stats_per_class = process_all_images()
print(f"\n{'Dimensiuni imagini:':<25} {Counter(image_sizes).most_common(1)[0][0]}")

print("-" * 60)
print("Statistici pixeli per clasă (Media ± Deviație Standard):")
print("-" * 60)

for class_idx in range(1, CFG.num_classes + 1):
    stats = pixel_stats_per_class[class_idx]
    print(f"Clasa {class_idx:2d}: {stats['mean']:6.2f} ± {stats['std']:6.2f}  (min: {stats['min']:3.0f}, max: {stats['max']:3.0f})")

print("=" * 60)

In [ ]:
X, y = process_all_images(operation_type = 'correlation_purpose')
corr_matrix = np.corrcoef(X)

intra_correlations = []
inter_correlations = []
num_samples = len(y)

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

class_mask = (y[:, None] == y[None, :]) & mask

intra_correlations = corr_matrix[class_mask]
inter_correlations = corr_matrix[mask & ~class_mask]

avg_intra = np.mean(intra_correlations)
avg_inter = np.mean(inter_correlations)

print(f"Average Intra-class Correlation (Consistency): {avg_intra:.4f}")
print(f"Average Inter-class Correlation (Confusion):   {avg_inter:.4f}")
print(f"Separability Score (Intra - Inter):            {avg_intra - avg_inter:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.kdeplot(intra_correlations, label='Intra-class', fill=True, color='green')
sns.kdeplot(inter_correlations, label='Inter-class', fill=True, color='red')
plt.xlabel('Correlation Coefficient')
plt.legend()

In [ ]:
_, _ = process_all_images(operation_type = 'plot_images')

In [ ]:
def make_train_and_test_csvs(train=True):
    data, _ = process_all_images(operation_type='make_csv_files', train=train)
    return pd.DataFrame(data)

In [ ]:
def locate_csv_to_folders(train=True):
    df = make_train_and_test_csvs(train=train)
    split_name = "train" if train else "test"
    csv_path = os.path.join(set_path(train), f"{split_name}_data.csv")
    df.to_csv(csv_path, index=False)

In [ ]:
locate_csv_to_folders(train=True)
locate_csv_to_folders(train=False)

In [ ]:
train_set = pd.read_csv(f'{CFG.train_path}/train_data.csv')
test_set = pd.read_csv(f'{CFG.test_path}/test_data.csv')

In [ ]:
train_set, val_set = train_test_split(
    train_set, 
    test_size=0.20, 
    stratify=train_set['label'], 
    random_state=CFG.seed
)

In [ ]:
def return_images_per_label(df):
    label_dictionary_of_images = {}
    for row in df.itertuples():
        label = row.ID.split('/')[-2]
        if label not in label_dictionary_of_images:
            label_dictionary_of_images[label] = []

        label_dictionary_of_images[label].append(row.ID)

    return label_dictionary_of_images

In [ ]:
augmentation_transform = transforms.Compose([
    transforms.RandomRotation(degrees=(-8, 8)),
    transforms.RandomResizedCrop(size=(CFG.img_size, CFG.img_size), scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.05, contrast=0.05)
])

train_images_by_label = return_images_per_label(train_set)

target_count = 1200

for label, train_image_paths in train_images_by_label.items():
    folder_name = str(label)
    folder_path = os.path.join(CFG.train_path, folder_name)
    
    train_image_names = [os.path.basename(path) for path in train_image_paths]
    
    num_current = len(train_image_names)
    num_needed = target_count - num_current
    
    if num_needed > 0 and target_count > num_current:
        print(f"Class {label}: Found {num_current} training images. Generating {num_needed} synthetic...")
        
        generated = 0
        while generated < num_needed:
            source_img_name = random.choice(train_image_names)
            source_img_path = os.path.join(folder_path, source_img_name)
            
            image = Image.open(source_img_path).convert('RGB')
            image = transforms.Resize((CFG.img_size, CFG.img_size))(image)
            
            augmented_image = augmentation_transform(image)
            
            aug_filename = f'aug_{generated}_{source_img_name}'
            aug_path = os.path.join(folder_path, aug_filename)
            augmented_image.save(aug_path)
            
            generated += 1
            
            if (generated % 100) == 0:
                print(f"  Generated {generated}/{num_needed} images for class {label}")
    else:
        print(f"Class {label}: Already has {num_current} images (>= {target_count})")
                
print(f"\nSynthetic data generation complete!")
print(f"Training set now has ~{target_count} images per class")
print(f"Validation set remains unchanged with original images only")

In [ ]:
train_data_with_synthetic = []
for class_idx in range(1, CFG.num_classes + 1):
    class_folder = os.path.join(CFG.train_path, str(class_idx))
    all_images = [f for f in os.listdir(class_folder) if f.endswith(CFG.image_extension)]
    
    for img_name in all_images:
        img_path = os.path.join(class_folder, img_name)
        train_data_with_synthetic.append({'ID': img_path, 'label': class_idx})

train_set_full = pd.DataFrame(train_data_with_synthetic)

train_image_basenames = set()
for row in train_set.itertuples():
    train_image_basenames.add(os.path.basename(row.ID))

In [ ]:
train_set_final = []
for row in train_set_full.itertuples():
    img_basename = os.path.basename(row.ID)
    if img_basename in train_image_basenames or img_basename.startswith('aug_'):
        train_set_final.append({'ID': row.ID, 'label': row.label})

train_set = pd.DataFrame(train_set_final)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=CFG.early_stopping_steps, verbose=False, path='checkpoint.pt', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = -np.inf
        self.early_stop = False
        self.path = path
        self.trace_func = trace_func

    def __call__(self, current_accuracy, model):
        if current_accuracy <= self.best_score + 1e-6:
            self.counter += 1
            if self.verbose:
                self.trace_func(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            if self.verbose:
                self.trace_func(
                    f"Validation accuracy increased ({self.best_score:.6f} → {current_accuracy:.6f}). Saving model..."
                )
            self.save_checkpoint(model)
            self.best_score = current_accuracy
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)


In [ ]:
class ImprovedMLP(nn.Module):
    def __init__(self, input_size = CFG.number_of_channels * CFG.img_size * CFG.img_size, 
                 hidden_sizes=[512, 256, 128], num_classes=10, dropout_rate=0.35):
        super(ImprovedMLP, self).__init__()
        
        layers = []
        in_features = input_size
        
        for i, hidden in enumerate(hidden_sizes):
            layers.append(nn.Linear(in_features, hidden))
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.ReLU())
            dropout = dropout_rate if i < len(hidden_sizes) - 1 else dropout_rate * 0.5
            layers.append(nn.Dropout(dropout))
            in_features = hidden
            
        layers.append(nn.Linear(in_features, num_classes))
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x)

In [ ]:
class MyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

        self.resize = transforms.Resize((CFG.img_size, CFG.img_size))
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=CFG.imagenet_mean, std=CFG.imagenet_std)
        ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx): 
        img_name = self.df.iloc[idx]['ID']
        label = self.df.iloc[idx]['label']
        image = Image.open(f'{img_name}').convert('RGB')

        label = label - 1  

        if self.transform:
            image = self.resize(image)
            image = self.transform(image)
            image = self.normalize(image)
        else:
            image = self.resize(image)
            image = self.normalize(image)
        
        return image, label


train_transform = transforms.Compose([
    transforms.RandomCrop(size=(CFG.img_size, CFG.img_size), padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=(-8, 8)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1)
])

train_dataset = MyDataset(train_set, transform=train_transform)
validation_dataset = MyDataset(val_set)
test_dataset = MyDataset(test_set)

train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=CFG.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=False)

In [ ]:
def get_number_of_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def top_k_accuracy(k, target, output, device):
    batch_size = target.size(0)
    
    _, pred = output.topk(k, 1, True, True)
    
    pred = pred.t()
    
    correct = pred.eq(target.to(device).view(1, -1).expand_as(pred))
    
    correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
    correct_k = correct_k.mul_(100.0 / batch_size)
    
    return correct_k.item()

In [ ]:
def plot_loss_curves(train_losses, val_losses):
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, 'b-o', label='Training Loss', linewidth=2, markersize=5)
    plt.plot(epochs, val_losses, 'r-o', label='Validation Loss', linewidth=2, markersize=5)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    plt.title('Curbe de Loss pentru Antrenare și Validare', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_accuracy_curves(train_top1, val_top1):
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_top1) + 1)
    
    plt.plot(epochs, train_top1, 'b-o', label='Training Top-1 Accuracy', linewidth=2, markersize=5)
    plt.plot(epochs, val_top1, 'r-o', label='Validation Top-1 Accuracy', linewidth=2, markersize=5)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Acuratețe (%)', fontsize=14)
    plt.title('Curbe de Acuratețe pentru Antrenare și Validare', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=range(1, CFG.num_classes + 1),
                yticklabels=range(1, CFG.num_classes + 1),
                cbar_kws={'label': 'Număr de predicții'})
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.title('Matricea de Confuzie', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return cm

In [ ]:
def validate(model, criterion, val_loader, device, epoch):
    model.eval()
    running_loss = 0.0
    running_top1_acc = 0.0
    
    bar = tqdm(enumerate(val_loader), total=len(val_loader), colour='green', file=sys.stdout)
    
    with torch.no_grad():
        for i, (images, labels) in bar:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels.long())
            
            top1_acc = top_k_accuracy(1, labels, outputs, device)
            
            running_loss += loss.item() * images.size(0)
            running_top1_acc += top1_acc * images.size(0)
            
            avg_loss = running_loss / ((i + 1) * images.size(0))
            avg_top1 = running_top1_acc / ((i + 1) * images.size(0))
            
            bar.set_postfix({
                'epoch': epoch,
                'val_loss': f'{avg_loss:.4f}',
                'top1_accuracy': f'{avg_top1:.2f}%'
            })
    
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_top1 = running_top1_acc / len(val_loader.dataset)
    
    return epoch_loss, epoch_top1

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
def train_single_model(config, train_loader, val_loader, device, num_epochs=CFG.epochs):
    model = ImprovedMLP().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], 
                           weight_decay=config['weight_decay'], betas=(0.9, 0.999))
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = CFG.criterion
    early_stopping = EarlyStopping(patience=CFG.early_stopping_steps, verbose=True, path='best_model.pth')
    
    train_losses, train_top1 = [], []
    val_losses, val_top1 = [], []
    
    best_val_acc = 0.0
    
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        running_top1_acc = 0.0
        
        bar = tqdm(enumerate(train_loader), total=len(train_loader), colour='cyan', file=sys.stdout)

        for i, (images, labels) in bar:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss = criterion(outputs, labels.long())
            loss.backward()
            optimizer.step()
            
            top1_acc = top_k_accuracy(1, labels, outputs, device)
            
            running_loss += loss.item() * images.size(0)
            running_top1_acc += top1_acc * images.size(0)

            avg_loss = running_loss / ((i + 1) * images.size(0))
            avg_top1 = running_top1_acc / ((i + 1) * images.size(0))

            bar.set_postfix({
            'epoch': epoch,
            'train_loss': f'{avg_loss:.4f}',
            'top1_accuracy': f'{avg_top1:.2f}%'
            })
            
        scheduler.step()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_top1 = running_top1_acc / len(train_loader.dataset)

        print(f"Train - Loss: {epoch_loss:.4f}, Top-1: {epoch_top1:.2f}%")

        train_losses.append(epoch_loss)
        train_top1.append(epoch_top1)
        
        val_loss, val_acc = validate(model, criterion, val_loader, device, epoch)

        print(f"Val - Loss: {val_loss:.4f}, Val - Top-1: {val_acc:.2f}%")

        val_losses.append(val_loss)
        val_top1.append(val_acc)
        
        early_stopping(val_acc, model)
        
        if early_stopping.early_stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc

        if epoch % CFG.steps_until_plot == 0:
            plot_loss_curves(train_losses, val_losses)
            plot_accuracy_curves(train_top1, val_top1)

    return model, best_val_acc

In [ ]:
def predict_soft_vote_ensemble(models, loader, device):
    for m in models:
        m.eval()
        
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            avg_probs = torch.zeros(images.size(0), CFG.num_classes).to(device)
            
            for model in models:
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                avg_probs += probs
                
            avg_probs /= len(models)
            
            _, predicted = torch.max(avg_probs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return np.array(all_preds), np.array(all_labels)

In [ ]:
def predict_hard_voting_ensemble(models, loader, device):
    for m in models:
        m.eval()
        
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            model_predictions  = []
            
            for model in models:
                outputs = model(images)
                _, predicted = outputs.max(1)
                model_predictions.append(predicted.cpu().numpy())
                
            model_predictions = np.array(model_predictions)
            
            ensemble_pred = []
            for i in range(model_predictions.shape[1]):
                votes = model_predictions[:, i]
                majority = Counter(votes).most_common(1)[0][0]
                ensemble_pred.append(majority)
            
            all_preds.extend(ensemble_pred)
            all_labels.extend(labels.cpu().numpy())
            
    return np.array(all_preds), np.array(all_labels)

In [ ]:
def apply_tta_augmentations(img_tensor):
    augmentations = []
    
    augmentations.append(img_tensor)
    
    augmentations.append(torch.flip(img_tensor, dims=[2]))
    
    return augmentations

In [ ]:
def predict_ensemble_with_tta(models, loader, device, num_tta=2):
    for m in models:
        m.eval()
        
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="TTA Ensemble Prediction", colour='magenta'):
            images, labels = images.to(device), labels.to(device)
            batch_size = images.size(0)
            
            final_probs = torch.zeros(batch_size, CFG.num_classes).to(device)
            
            for img_idx in range(batch_size):
                img = images[img_idx]
                
                
                augmented_images = apply_tta_augmentations(img)
                augmented_images = augmented_images[:min(num_tta, 2)]
                
                aug_batch = torch.stack(augmented_images)
                
                tta_probs = torch.zeros(CFG.num_classes).to(device)
                
                for model in models:
                    outputs = model(aug_batch)
                    probs = F.softmax(outputs, dim=1)
                    avg_probs = probs.mean(dim=0)
                    tta_probs += avg_probs
                
                tta_probs /= len(models)
                final_probs[img_idx] = tta_probs
            
            _, predicted = torch.max(final_probs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return np.array(all_preds), np.array(all_labels)


In [ ]:
def calculate_f1_score(y_true, y_pred):
    return f1_score(y_true, y_pred, average='weighted') * 100

In [ ]:
ensemble_configs = [
    {'lr': 6e-4, 'weight_decay': 1e-4, 'lr_decay': 0.93},
    {'lr': 8e-4, 'weight_decay': 8e-5, 'lr_decay': 0.94}, 
    {'lr': 5e-4, 'weight_decay': 3e-4, 'lr_decay': 0.92},
    {'lr': 7e-4, 'weight_decay': 9e-5, 'lr_decay': 0.935},
    {'lr': 5e-4, 'weight_decay': 1e-4, 'lr_decay': 0.915}
]

trained_models = []
model_accuracies = []

print(f"Starting Ensemble Training of {len(ensemble_configs)} models...\n")

for i, conf in enumerate(ensemble_configs):
    print(f"\n{'='*60}")
    print(f"Training Model {i+1}/{len(ensemble_configs)}")
    print(f"Config: lr={conf['lr']}, weight_decay={conf['weight_decay']}, lr_decay={conf['lr_decay']}")
    print(f"{'='*60}\n")
    
    model, best_acc = train_single_model(conf, train_loader, validation_loader, CFG.device)
    trained_models.append(model)
    model_accuracies.append(best_acc)
    
    print(f"\nModel {i+1} Best Validation Accuracy: {best_acc:.2f}%")

print(f"\n{'='*60}")
print("Ensemble Training Complete!")
print(f"{'='*60}")
print("\nIndividual Model Accuracies:")
for i, acc in enumerate(model_accuracies):
    print(f"  Model {i+1}: {acc:.2f}%")
print(f"\nAverage Accuracy: {np.mean(model_accuracies):.2f}%")

In [ ]:
combined = list(zip(trained_models, model_accuracies))
sorted_models = sorted(combined, key=lambda x: x[1], reverse=True)
top_three = sorted_models[:3]
top_three_models = [m for m, _ in top_three]
print("Top-3 validation accuracies:", [acc for _, acc in top_three])

In [ ]:
print("\nEvaluating Ensemble with TTA...")
for i in range(2,3):
    print(f" - Using {i} TTA augmentations...")
    y_pred_ens_tta, y_true_ens_tta = predict_ensemble_with_tta(top_three_models, test_loader, CFG.device, num_tta=i)
    
    ens_tta_acc_i = accuracy_score(y_true_ens_tta, y_pred_ens_tta) * 100
    ens_tta_f1_i = f1_score(y_true_ens_tta, y_pred_ens_tta, average='weighted') * 100
    
    print(f"   Ensemble with {i} TTA Accuracy: {ens_tta_acc_i:.2f}%")
    print(f"   Ensemble with {i} TTA F1 Score: {ens_tta_f1_i:.2f}%")

print("\nEvaluating Baseline Ensemble (without TTA)...")
y_pred_ensemble, y_true_ensemble = predict_soft_vote_ensemble(trained_models, test_loader, CFG.device)

ens_acc = accuracy_score(y_true_ensemble, y_pred_ensemble) * 100
ens_f1 = f1_score(y_true_ensemble, y_pred_ensemble, average='weighted') * 100

print(f"Baseline Ensemble Accuracy: {ens_acc:.2f}%")
print(f"Baseline Ensemble F1 Score: {ens_f1:.2f}%")

In [ ]:
print("\nEvaluating Ensemble...")
y_pred_hard_ensemble, y_true_hard_ensemble = predict_hard_voting_ensemble(top_three_models, test_loader, CFG.device)

ens_acc = accuracy_score(y_true_hard_ensemble, y_pred_hard_ensemble) * 100
ens_f1 = f1_score(y_true_hard_ensemble, y_pred_hard_ensemble, average='weighted') * 100

print(f"Ensemble Accuracy: {ens_acc:.2f}%")
print(f"Ensemble F1 Score: {ens_f1:.2f}%")

print("\nGenerating Ensemble Confusion Matrix...")
plot_confusion_matrix(y_true_hard_ensemble, y_pred_hard_ensemble)